In [1]:
# Task 3: Correlation between News Sentiment and Stock Movement Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from datetime import datetime, timedelta
import os
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Notebook settings
%matplotlib inline
plt.style.use('seaborn')
pd.set_option('display.max_columns', None)


ModuleNotFoundError: No module named 'textblob'

In [ ]:
# 1. Load and prepare data
def load_stock_data(ticker):
    """Load stock data for a given ticker"""
    base_path = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))), 
                            "data", "raw", "yfinance_data")
    file_path = os.path.join(base_path, f"{ticker}_historical_data.csv")
    df = pd.read_csv(file_path, parse_dates=['Date'])
    df.set_index('Date', inplace=True)
    return df

def load_news_data():
    """Load news data"""
    base_path = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))), 
                            "data", "raw")
    file_path = os.path.join(base_path, "raw_analyst_ratings.csv")
    df = pd.read_csv(file_path, parse_dates=['date'])
    return df

# Load data
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT', 'NVDA', 'TSLA']
stock_data = {ticker: load_stock_data(ticker) for ticker in tickers}
news_df = load_news_data()

print("Data loaded successfully!")
print("\nStock data shape for each ticker:")
for ticker, df in stock_data.items():
    print(f"{ticker}: {df.shape}")
print(f"\nNews data shape: {news_df.shape}")


In [ ]:
# 2. Sentiment Analysis Functions
def get_sentiment_score(text):
    """Calculate sentiment score using TextBlob"""
    try:
        return TextBlob(str(text)).sentiment.polarity
    except:
        return 0

def get_sentiment_label(score):
    """Convert sentiment score to label"""
    if score > 0.1:
        return 'Positive'
    elif score < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

# Apply sentiment analysis to headlines
news_df['sentiment_score'] = news_df['headline'].apply(get_sentiment_score)
news_df['sentiment_label'] = news_df['sentiment_score'].apply(get_sentiment_label)

# Display sentiment distribution
sentiment_dist = news_df['sentiment_label'].value_counts()
print("Sentiment Distribution:")
print(sentiment_dist)

# Visualize sentiment distribution
plt.figure(figsize=(10, 6))
sentiment_dist.plot(kind='bar')
plt.title('Distribution of News Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Show sample headlines with their sentiment scores
print("\nSample Headlines and Their Sentiment Scores:")
sample_headlines = news_df[['headline', 'sentiment_score', 'sentiment_label']].sample(5)
print(sample_headlines)


In [ ]:
# 3. Calculate Daily Stock Returns and Prepare Data for Correlation Analysis
def calculate_returns_and_sentiment(stock_df, news_df, ticker):
    """Calculate daily returns and aggregate sentiment scores"""
    # Calculate daily returns
    stock_df['returns'] = stock_df['Close'].pct_change()
    
    # Filter news for the specific ticker
    ticker_news = news_df[news_df['stock'] == ticker].copy()
    
    # Aggregate daily sentiment
    daily_sentiment = ticker_news.groupby(ticker_news['date'].dt.date).agg({
        'sentiment_score': ['mean', 'count']
    }).reset_index()
    daily_sentiment.columns = ['date', 'avg_sentiment', 'news_count']
    
    # Convert date to datetime for merging
    daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])
    
    # Merge with stock data
    merged_df = pd.merge(
        stock_df.reset_index(),
        daily_sentiment,
        left_on=stock_df.index.date,
        right_on='date',
        how='left'
    )
    
    # Fill missing sentiment values with 0 (days without news)
    merged_df['avg_sentiment'].fillna(0, inplace=True)
    merged_df['news_count'].fillna(0, inplace=True)
    
    return merged_df

# Process each ticker
correlation_results = []
for ticker in tickers:
    # Calculate returns and merge with sentiment
    merged_data = calculate_returns_and_sentiment(stock_data[ticker], news_df, ticker)
    
    # Calculate correlation
    correlation = stats.pearsonr(
        merged_data['returns'].fillna(0),
        merged_data['avg_sentiment']
    )
    
    correlation_results.append({
        'ticker': ticker,
        'correlation': correlation[0],
        'p_value': correlation[1]
    })

# Display correlation results
correlation_df = pd.DataFrame(correlation_results)
print("Correlation between Daily Returns and News Sentiment:")
print(correlation_df)

# Visualize correlations
plt.figure(figsize=(10, 6))
sns.barplot(x='ticker', y='correlation', data=correlation_df)
plt.title('Correlation between Stock Returns and News Sentiment')
plt.xlabel('Stock')
plt.ylabel('Correlation Coefficient')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Detailed Analysis for Individual Stocks
def analyze_stock_sentiment_relationship(ticker, stock_df, news_df):
    """Perform detailed analysis for a single stock"""
    # Get merged data
    merged_data = calculate_returns_and_sentiment(stock_df, news_df, ticker)
    
    # Create subplots
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    fig.suptitle(f'Sentiment Analysis for {ticker}', fontsize=16)
    
    # 1. Time series of returns and sentiment
    ax1 = axes[0]
    ax1.plot(merged_data['Date'], merged_data['returns'], 
            label='Returns', alpha=0.5)
    ax1.plot(merged_data['Date'], merged_data['avg_sentiment'], 
            label='Sentiment', alpha=0.5)
    ax1.set_title('Daily Returns vs Sentiment Score')
    ax1.legend()
    ax1.set_xlabel('Date')
    
    # 2. Scatter plot
    ax2 = axes[1]
    ax2.scatter(merged_data['avg_sentiment'], merged_data['returns'], 
               alpha=0.5)
    ax2.set_title('Returns vs Sentiment Score')
    ax2.set_xlabel('Sentiment Score')
    ax2.set_ylabel('Returns')
    
    # Add trend line
    z = np.polyfit(merged_data['avg_sentiment'], 
                   merged_data['returns'], 1)
    p = np.poly1d(z)
    ax2.plot(merged_data['avg_sentiment'], 
             p(merged_data['avg_sentiment']), "r--", alpha=0.8)
    
    # 3. News count histogram
    ax3 = axes[2]
    ax3.hist(merged_data['news_count'][merged_data['news_count'] > 0], 
             bins=30)
    ax3.set_title('Distribution of Daily News Count')
    ax3.set_xlabel('Number of News Articles')
    ax3.set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # Calculate additional statistics
    stats_dict = {
        'Total News Articles': len(news_df[news_df['stock'] == ticker]),
        'Days with News': len(merged_data[merged_data['news_count'] > 0]),
        'Avg News per Day': merged_data['news_count'].mean(),
        'Max News in One Day': merged_data['news_count'].max(),
        'Avg Sentiment Score': merged_data['avg_sentiment'].mean(),
        'Sentiment Std Dev': merged_data['avg_sentiment'].std()
    }
    
    print(f"\nDetailed Statistics for {ticker}:")
    for key, value in stats_dict.items():
        print(f"{key}: {value:.2f}")

# Analyze each stock (example with one stock)
analyze_stock_sentiment_relationship('AAPL', stock_data['AAPL'], news_df)
